# IALS (Implicit Alternating Least Squares )

В этом ноутбуке я построю IALS модель, протестирую её и сохраню 

## Импорт библиотек 

In [39]:
import pandas as pd
import numpy as np
import scipy 
import os
import implicit 
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from implicit.evaluation import train_test_split as implicit_split 
from implicit.evaluation import precision_at_k, mean_average_precision_at_k, ranking_metrics_at_k
import implicit.gpu
import optuna
import optuna.visualization as vis
import plotly
import time


In [26]:
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

## Загрузка данных

#### Выгружаем матрицу

In [27]:
results_path = Path("../../../results/matrices")
user_item_sparse = scipy.sparse.load_npz(results_path/'artnail_user_item_sparse.npz')

#### Выгружаем очищенные таблицы

In [28]:
clean_path = Path("../../../Tables/CleanTable")
users_clean = pd.read_csv(clean_path / 'users_clean_final.csv')
items_clean = pd.read_csv(clean_path / 'items_clean_final.csv')

In [29]:
f'Размеры матрицы: {user_item_sparse.shape}'

'Размеры матрицы: (2893, 258)'

## IALS model 

#### Train/Validation split для метрик

In [30]:
train_sparse, val_sparse = implicit_split(user_item_sparse, train_percentage=0.9, random_state=42)

In [31]:
print(f"  Train: {train_sparse.nnz} ({train_sparse.nnz/user_item_sparse.nnz:.1%})")
print(f"  Val:   {val_sparse.nnz} ({val_sparse.nnz/user_item_sparse.nnz:.1%})")

  Train: 877 (90.4%)
  Val:   93 (9.6%)


#### Построение базовой модели 

In [32]:
IALS_model = implicit.als.AlternatingLeastSquares(
    factors=64,           # Размер эмбеддингов (пользователи × услуги)
    regularization=0.1,   # L2 регуляризация (0.01-0.5)
    iterations=50,        # ALS итераций (сходимость)
    random_state=42,      # Репродуцируемость
    use_gpu=False,        # CPU 
    num_threads=4         # Потоки (4-8 оптимально)
)

Обучение 

In [33]:
CONFIDENCE_SCALE = 15
IALS_model.fit(train_sparse*CONFIDENCE_SCALE)

print(f"  User embeddings: {IALS_model.user_factors.shape}")
print(f"  Item embeddings: {IALS_model.item_factors.shape}")

  0%|          | 0/50 [00:00<?, ?it/s]

  User embeddings: (2893, 64)
  Item embeddings: (258, 64)


#### Оценка качества базовой модели 

IALS будет выполнять функции подбора кандидатов, а функцию ранжирования будет выполнять `CutBoost` модель, поэтому в оценке `IALS` главной метрикой буду считать Recall@K, где K будет большим.

Цель не отобрать конкретные 10 лучших вариантов, а отобрать 50 кандидатов, главное чтобы среди них были хорошие кандидаты и CutBoost увидел их

In [34]:
metrics = ranking_metrics_at_k(IALS_model, train_sparse, val_sparse, K=50)
metrics

  0%|          | 0/91 [00:00<?, ?it/s]

{'precision': 0.5806451612903226,
 'map': 0.029722000906388105,
 'ndcg': 0.12825269952781929,
 'auc': 0.6866756822032526}

Пояснение: В implicit версии 0.7.2 метрика Precision@K считает по факту как Recall@K, поэтому мы будем интарпретировать precision как recall. 

In [35]:
recall_50 = metrics['precision']
f'Recall@50 = {recall_50:.6f}'

'Recall@50 = 0.580645'

__Из 50 кандидатов 58% релевантны__. Это хороший показатель для CutBoost. 

Также  `AUC` = 0.6819 показывает, что модель уверенно отличает положительные и негативные оценки.

Из остальных метрик можно понять, что модель плохо ранжирует отобранных кандидатов, но для нашей ситуации это не страшно, т. к. ранжировать кандидатов будет CutBoost.

#### Random Search с  помощью Optuna

In [67]:

def objective(trial):
    """Optuna objective: максимизируем Recall@50"""
    
    # Предлагаем параметры
    params = {
    # Размер эмбеддингов: от компактных до довольно крупных
    # (чем больше factors, тем качественнее, но медленнее и больше памяти)
    'factors': trial.suggest_int('factors', 32, 160, step=16),  
    # 32, 48, 64, 80, 96, 112, 128, 144, 160

    # Регуляризация: от очень слабой до довольно сильной
    # (я бы уже использовал log-шкалу — так лучше исследуются края)
    'regularization': trial.suggest_float('regularization', 0.01, 0.5),

    # Число итераций: чуть меньше базового до заметно больше
    # (если данные не очень большие, можно позволить до 80)
    'iterations': trial.suggest_int('iterations', 30, 80),

    # Дополнительные параметры ALS:
    # weight для нормализации регуляризации по числу факторов
    'dtype': np.float32,  # чтобы каждый trial был одинаков по типу
    'use_gpu': False,
    'num_threads': 4,
    'random_state': 42,
    'calculate_training_loss': False
}

    # Модель
    model = implicit.als.AlternatingLeastSquares(**params)
    
    # Обучение 
    CONFIDENCE_SCALE = 15
    model.fit(train_sparse*CONFIDENCE_SCALE, show_progress=False)

    # Метрики
    metrics = ranking_metrics_at_k(
        model, 
        train_user_items=train_sparse, 
        test_user_items=val_sparse, 
        K=50,
        show_progress=False
    )

    recall50 = metrics['precision']

     
    # Callback для pruning (Optuna может прервать плохой trial)
    trial.report(recall50, step=1)
    if trial.should_prune():
        raise optuna.TrialPruned()
    
    
    print(f"✅ Trial {trial.number}: Recall@50={recall50:.4f} | {params}")
    return recall50


In [68]:

study = optuna.create_study(
    study_name='IALS Optuna Optimization',
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(),
    storage='sqlite:///ex.db',
    load_if_exists=True
)

[I 2026-03-09 22:01:11,697] Using an existing study with name 'IALS Optuna Optimization' instead of creating a new one.


In [69]:
study.optimize(objective, n_trials=50)
print(f"Лучший Recall@50: {study.best_value}")


[I 2026-03-09 22:01:13,206] Trial 32 pruned. 
[I 2026-03-09 22:01:14,506] Trial 33 pruned. 
[I 2026-03-09 22:01:16,476] Trial 34 pruned. 
[I 2026-03-09 22:01:18,313] Trial 35 finished with value: 0.6344086021505376 and parameters: {'factors': 160, 'regularization': 0.4480284162027014, 'iterations': 47}. Best is trial 35 with value: 0.6344086021505376.


✅ Trial 35: Recall@50=0.6344 | {'factors': 160, 'regularization': 0.4480284162027014, 'iterations': 47, 'dtype': <class 'numpy.float32'>, 'use_gpu': False, 'num_threads': 4, 'random_state': 42, 'calculate_training_loss': False}


[I 2026-03-09 22:01:20,021] Trial 36 finished with value: 0.6344086021505376 and parameters: {'factors': 160, 'regularization': 0.47014460516558587, 'iterations': 44}. Best is trial 35 with value: 0.6344086021505376.


✅ Trial 36: Recall@50=0.6344 | {'factors': 160, 'regularization': 0.47014460516558587, 'iterations': 44, 'dtype': <class 'numpy.float32'>, 'use_gpu': False, 'num_threads': 4, 'random_state': 42, 'calculate_training_loss': False}


[I 2026-03-09 22:01:21,326] Trial 37 finished with value: 0.6129032258064516 and parameters: {'factors': 160, 'regularization': 0.4957972998836967, 'iterations': 33}. Best is trial 35 with value: 0.6344086021505376.


...

✅ Trial 79: Recall@50=0.6559 | {'factors': 96, 'regularization': 0.48181303986280566, 'iterations': 36, 'dtype': <class 'numpy.float32'>, 'use_gpu': False, 'num_threads': 4, 'random_state': 42, 'calculate_training_loss': False}


[I 2026-03-09 22:03:12,056] Trial 80 finished with value: 0.6451612903225806 and parameters: {'factors': 96, 'regularization': 0.4636099840657895, 'iterations': 36}. Best is trial 44 with value: 0.6559139784946236.


✅ Trial 80: Recall@50=0.6452 | {'factors': 96, 'regularization': 0.4636099840657895, 'iterations': 36, 'dtype': <class 'numpy.float32'>, 'use_gpu': False, 'num_threads': 4, 'random_state': 42, 'calculate_training_loss': False}


[I 2026-03-09 22:03:14,706] Trial 81 finished with value: 0.6451612903225806 and parameters: {'factors': 80, 'regularization': 0.48543333330613886, 'iterations': 38}. Best is trial 44 with value: 0.6559139784946236.


✅ Trial 81: Recall@50=0.6452 | {'factors': 80, 'regularization': 0.48543333330613886, 'iterations': 38, 'dtype': <class 'numpy.float32'>, 'use_gpu': False, 'num_threads': 4, 'random_state': 42, 'calculate_training_loss': False}
Лучший Recall@50: 0.6559139784946236


In [70]:
best_params = study.best_params
f'best_params = {best_params}'

"best_params = {'factors': 144, 'regularization': 0.4501461402788599, 'iterations': 39}"